In [ ]:
!pip install --quiet faiss-gpu rapidfuzz xgboost cupy-cuda12x psutil cudf-cu12 dask-cudf-cu12 --extra-index-url=https://pypi.nvidia.com

import os, gc, re, csv, zipfile, sqlite3, urllib.request, psutil
import numpy as np, pandas as pd, xgboost as xgb
import cupy as cp, cupyx.scipy.sparse as cpx_sparse
import faiss
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler
from sklearn.feature_extraction.text import HashingVectorizer

try:
    import cudf
    USE_CUDF = True
    print("cuDF loaded successfully! GPU will be used for ETL.")
except ImportError:
    USE_CUDF = False
    print("cuDF not found. Falling back to CPU Pandas.")

for d in ["dataset/train", "dataset/test", "output", "utils"]: os.makedirs(d, exist_ok=True)
print("Environment setup complete.")

cuDF loaded successfully! GPU will be used for ETL.
Environment setup complete.


In [ ]:
DATA_URL = "https://cdn.unstop.com/files/6ab10eb3b23ba_student_resource.zip"
if not os.path.exists("dataset/train/train_source1.tsv"):
    print("Downloading 30M row dataset...")
    urllib.request.urlretrieve(DATA_URL, "dataset.zip")
    with zipfile.ZipFile("dataset.zip", "r") as z: z.extractall("temp_data")
    import shutil
    for root, _, files in os.walk("temp_data"):
        for f in files:
            if f.endswith(".tsv"):
                folder = "train" if "train" in root.lower() or "train" in f.lower() else "test"
                shutil.copy(os.path.join(root, f), os.path.join(f"dataset/{folder}", f))
    shutil.rmtree("temp_data"); os.remove("dataset.zip")
    print("Data downloaded and extracted.")
else:
    print("Data already exists. Skipping download.")

Data already exists. Skipping download.


In [ ]:
DB_PATH = "pool_data.db"

def clean_text(text):
    if not isinstance(text, str): return ""
    return re.sub(r"\s+", " ", re.sub(r"[^\w\s]", " ", text.lower())).strip()

if not os.path.exists(DB_PATH):
    print("Building Out-of-Core SQLite Database for Pool Records...")
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute("PRAGMA synchronous = OFF"); c.execute("PRAGMA journal_mode = MEMORY")
    c.execute("CREATE TABLE pool (entity_id TEXT PRIMARY KEY, country TEXT, name TEXT, addr TEXT, combined TEXT, nums TEXT)")

    for file in ["dataset/train/train_source2.tsv", "dataset/train/train_source3.tsv",
                 "dataset/test/test_source2.tsv", "dataset/test/test_source3.tsv"]:
        print(f"Streaming {file} into DB...")
        for chunk_pd in pd.read_csv(file, sep="\t", dtype=str, chunksize=500000):
            if USE_CUDF:
                gdf = cudf.from_pandas(chunk_pd)
                gdf["country"] = gdf["country"].fillna("UNKNOWN").str.strip().str.upper()
                gdf["name"] = gdf["business_name"].fillna("").str.lower().str.replace(r"[^\w\s]", " ", regex=True).str.replace(r"\s+", " ", regex=True).str.strip()
                gdf["addr"] = gdf["business_address"].fillna("").str.lower().str.replace(r"[^\w\s]", " ", regex=True).str.replace(r"\s+", " ", regex=True).str.strip()
                gdf["combined"] = gdf["name"] + " " + gdf["addr"]
                chunk_pd = gdf.to_pandas()
                del gdf
            else:
                chunk_pd["country"] = chunk_pd["country"].fillna("UNKNOWN").str.strip().str.upper()
                chunk_pd["name"] = chunk_pd["business_name"].apply(clean_text)
                chunk_pd["addr"] = chunk_pd["business_address"].apply(clean_text)
                chunk_pd["combined"] = chunk_pd["name"] + " " + chunk_pd["addr"]

            chunk_pd["nums"] = chunk_pd["addr"].apply(lambda x: ",".join(re.findall(r"\b\d+\b", x)))
            c.executemany("INSERT OR IGNORE INTO pool VALUES (?,?,?,?,?,?)", chunk_pd[["entity_id", "country", "name", "addr", "combined", "nums"]].values.tolist())

    c.execute("CREATE INDEX idx_country ON pool(country)")
    conn.commit(); conn.close()
    print("Database built successfully!")
else:
    print("Database already exists. Skipping construction.")

Database already exists. Skipping construction.


In [ ]:
VECTOR_DIM = 64  # FIXED: 64 dimensions * 10M records = ~2.5 GB VRAM!

# 1. Base Sparse Hash
vectorizer = HashingVectorizer(analyzer="char_wb", ngram_range=(3,3), n_features=2**15, dtype=np.float32)

# 2. Random Projection Matrix for Locality Sensitive Hashing (LSH)
np.random.seed(42)
R_PROJECTION = np.random.randn(2**15, VECTOR_DIM).astype(np.float32) / np.sqrt(VECTOR_DIM)

def text_to_dense_l2(texts):
    """Converts strings to sparse matrix, projects to dense 64d, and L2 normalizes for FAISS Cosine"""
    sparse_mat = vectorizer.transform(texts)
    dense_mat = np.ascontiguousarray(sparse_mat.dot(R_PROJECTION).astype(np.float32))
    faiss.normalize_L2(dense_mat) # In-place Normalization
    return dense_mat

def bulk_fetch_candidates(c, cand_ids):
    pool_dict = {}
    cand_list = list(set(cand_ids))
    for i in range(0, len(cand_list), 900):
        chunk = cand_list[i:i+900]
        c.execute(f"SELECT entity_id, name, addr, nums FROM pool WHERE entity_id IN ({','.join(['?']*len(chunk))})", chunk)
        for row in c.fetchall(): pool_dict[row[0]] = (row[1], row[2], row[3])
    return pool_dict

def extract_features(n1, a1, num1_str, n2, a2, num2_str):
    n1, a1, n2, a2 = n1 or "", a1 or "", n2 or "", a2 or ""
    nums1, nums2 = set(num1_str.split(",")) if num1_str else set(), set(num2_str.split(",")) if num2_str else set()
    nums1.discard(""); nums2.discard("")
    d_jac = len(nums1 & nums2) / len(nums1 | nums2) if nums1 and nums2 else 0.0
    d_match = 1.0 if nums1 and nums2 and (nums1 & nums2) else (0.5 if not nums1 or not nums2 else 0.0)
    return [fuzz.ratio(n1, n2)/100.0, fuzz.token_sort_ratio(n1, n2)/100.0, JaroWinkler.similarity(n1, n2), fuzz.ratio(a1, a2)/100.0, d_jac, d_match]

print("FAISS engines loaded successfully.")

FAISS engines loaded successfully.


In [ ]:
print("Loading previously trained XGBoost Model...")

# Initialize the model structure
model = xgb.XGBClassifier()

# Load your pre-trained weights
# Make sure "xgb_entity_model.json" is uploaded to your current directory!
model.load_model("xgb_entity_model.json")

print("✅ Model successfully loaded!")

Loading previously trained XGBoost Model...
✅ Model successfully loaded!


In [ ]:
import csv, gc, re, pandas as pd, numpy as np, sqlite3
import faiss

print("=== FINAL TEST INFERENCE (OPTIMIZED FOR F_0.5) ===")

# HYPERPARAMETERS
FAISS_TOP_K = 20           # Keeps candidate list small for ranking tie-breakers, but high enough for recall
FAISS_MIN_SIM = 0.20       # Filters absolute garbage before XGBoost
XGB_STRICT_THRESH = 0.93   # Highly strict to preserve singleton 1.0 scores
XGB_RESCUE_THRESH = 0.85   # Rescue borderline predictions if string match is near-perfect

conn = sqlite3.connect("pool_data.db")

# 1. Load valid Test IDs to prevent data leaks from the Training set
print("Loading valid test IDs for strict FAISS indexing...")
valid_s2 = pd.read_csv("dataset/test/test_source2.tsv", sep="\t", usecols=["entity_id"], dtype=str)
valid_s3 = pd.read_csv("dataset/test/test_source3.tsv", sep="\t", usecols=["entity_id"], dtype=str)
valid_test_ids = set(valid_s2["entity_id"].dropna()).union(set(valid_s3["entity_id"].dropna()))
del valid_s2, valid_s3; gc.collect()

# 2. Strict Test-Only FAISS Builder
def build_test_only_faiss_index(country, conn, valid_ids):
    c = conn.cursor()
    c.execute("SELECT COUNT(*) FROM pool WHERE country=?", (country,))
    total = c.fetchone()[0]

    cpu_index = faiss.IndexFlatIP(VECTOR_DIM)
    res = faiss.StandardGpuResources()
    res.noTempMemory()
    gpu_index = faiss.index_cpu_to_gpu(res, 0, cpu_index)

    global_ids = []
    chunk_size = 200000

    print(f"Building STRICT Test FAISS Index for {country}...")
    for offset in range(0, total, chunk_size):
        c.execute("SELECT entity_id, combined FROM pool WHERE country=? LIMIT ? OFFSET ?", (country, chunk_size, offset))
        data = c.fetchall()
        if not data: break

        valid_data = [row for row in data if row[0] in valid_ids]
        if not valid_data: continue

        global_ids.extend([row[0] for row in valid_data])
        dense_batch = text_to_dense_l2([row[1] for row in valid_data])
        gpu_index.add(dense_batch)

    return gpu_index, global_ids

def get_optimized_candidates(s1_chunk, index, global_ids):
    n_s1 = len(s1_chunk)
    final_ids = np.empty((n_s1, FAISS_TOP_K), dtype=object)
    if n_s1 == 0: return final_ids

    dense_query = text_to_dense_l2(s1_chunk["combined"].tolist())
    distances, indices = index.search(dense_query, FAISS_TOP_K)

    for i in range(n_s1):
        for j in range(FAISS_TOP_K):
            idx = indices[i, j]
            if idx != -1 and distances[i, j] >= FAISS_MIN_SIM:
                final_ids[i, j] = global_ids[idx]
            else:
                final_ids[i, j] = None
    return final_ids

# 3. Inference Loop
f_cand = open("output/candidate_pairs.tsv", "w", newline="")
f_match = open("output/matching_results.tsv", "w", newline="")
cw, mw = csv.writer(f_cand, delimiter="\t"), csv.writer(f_match, delimiter="\t")
cw.writerow(["source1_entity_id", "candidate_entity_ids"])
mw.writerow(["source1_entity_id", "matched_entity_ids"])

s1_test = pd.read_csv("dataset/test/test_source1.tsv", sep="\t", dtype=str)
s1_test["country"] = s1_test["country"].fillna("UNKNOWN").str.strip().str.upper()
s1_test["name"] = s1_test["business_name"].apply(clean_text)
s1_test["addr"] = s1_test["business_address"].apply(clean_text)
s1_test["combined"] = s1_test["name"] + " " + s1_test["addr"]
s1_test["nums"] = s1_test["addr"].apply(lambda x: ",".join(re.findall(r"\b\d+\b", x)))

model.set_params(device="cpu") # Avoid CUDA re-initialization conflicts

chunk_count = 0
for country in s1_test["country"].unique():
    s1_c = s1_test[s1_test["country"] == country].reset_index(drop=True)

    country_index, global_ids = build_test_only_faiss_index(country, conn, valid_test_ids)
    c = conn.cursor()

    print(f"Running inference for {country} ({len(s1_c):,} entities)...")
    for chunk_start in range(0, len(s1_c), 5000):
        chunk_count += 1
        chunk = s1_c.iloc[chunk_start:chunk_start+5000].reset_index(drop=True)

        top_cands = get_optimized_candidates(chunk, country_index, global_ids)

        all_cands = set()
        for i in range(len(chunk)): all_cands.update([cand for cand in top_cands[i] if cand is not None])
        pool_data_dict = bulk_fetch_candidates(c, list(all_cands))

        for i, s1_id in enumerate(chunk["entity_id"]):
            cands = [cand for cand in top_cands[i] if cand is not None]
            cw.writerow([s1_id, ",".join(cands)])

            if not cands:
                mw.writerow([s1_id, ""])
                continue

            X_test, valid_cands_list = [], []
            for cand_id in cands:
                if cand_id in pool_data_dict:
                    n2, a2, num2 = pool_data_dict[cand_id]
                    X_test.append(extract_features(chunk.at[i, "name"], chunk.at[i, "addr"], chunk.at[i, "nums"], n2, a2, num2))
                    valid_cands_list.append(cand_id)

            if X_test:
                probs = model.predict_proba(np.array(X_test))[:, 1]
                matches = []
                for idx, p in enumerate(probs):
                    feats = X_test[idx]
                    # feat[2] is Jaro-Winkler Name, feat[3] is Fuzz Addr
                    if p >= XGB_STRICT_THRESH:
                        matches.append(valid_cands_list[idx])
                    elif p >= XGB_RESCUE_THRESH and feats[2] >= 0.90 and feats[3] >= 0.85:
                        matches.append(valid_cands_list[idx])
                mw.writerow([s1_id, ",".join(matches)])
            else:
                mw.writerow([s1_id, ""])

        print(f"  -> Processed batch {chunk_count} ({(chunk_start + len(chunk)):,} / {len(s1_c):,} rows for {country})")
        gc.collect()

    del country_index, global_ids
    gc.collect()

f_cand.close(); f_match.close(); conn.close()
print("✅ Output streamed to output/ directory.")

=== FINAL TEST INFERENCE (OPTIMIZED FOR F_0.5) ===
Loading valid test IDs for strict FAISS indexing...
Building STRICT Test FAISS Index for US...
Running inference for US (663,106 entities)...
  -> Processed batch 1 (5,000 / 663,106 rows for US)
  -> Processed batch 2 (10,000 / 663,106 rows for US)
  -> Processed batch 3 (15,000 / 663,106 rows for US)
  -> Processed batch 4 (20,000 / 663,106 rows for US)
  -> Processed batch 5 (25,000 / 663,106 rows for US)
  -> Processed batch 6 (30,000 / 663,106 rows for US)
  -> Processed batch 7 (35,000 / 663,106 rows for US)
  -> Processed batch 8 (40,000 / 663,106 rows for US)
  -> Processed batch 9 (45,000 / 663,106 rows for US)
  -> Processed batch 10 (50,000 / 663,106 rows for US)
  -> Processed batch 11 (55,000 / 663,106 rows for US)
  -> Processed batch 12 (60,000 / 663,106 rows for US)
  -> Processed batch 13 (65,000 / 663,106 rows for US)
  -> Processed batch 14 (70,000 / 663,106 rows for US)
  -> Processed batch 15 (75,000 / 663,106 rows 

In [ ]:
import sys, os, pandas as pd, argparse
import zipfile

# 1. RUN VALIDATOR
validator_code = """
import sys, os, pandas as pd, argparse
def validate(matching_file, candidate_file, test_dir):
    issues = []
    s1_path, s2_path, s3_path = os.path.join(test_dir, "test_source1.tsv"), os.path.join(test_dir, "test_source2.tsv"), os.path.join(test_dir, "test_source3.tsv")
    valid_s1 = set(pd.read_csv(s1_path, sep="\\t")["entity_id"].dropna().astype(str))
    valid_pool = set(pd.read_csv(s2_path, sep="\\t")["entity_id"].dropna().astype(str)).union(set(pd.read_csv(s3_path, sep="\\t")["entity_id"].dropna().astype(str)))
    df_match = pd.read_csv(matching_file, sep="\\t", dtype=str, keep_default_na=False)
    df_cand = pd.read_csv(candidate_file, sep="\\t", dtype=str, keep_default_na=False)
    if list(df_match.columns) != ["source1_entity_id", "matched_entity_ids"]: issues.append("Bad Match Columns")
    if list(df_cand.columns) != ["source1_entity_id", "candidate_entity_ids"]: issues.append("Bad Cand Columns")
    if len(df_match["source1_entity_id"]) != len(set(df_match["source1_entity_id"])): issues.append("Match Duplicates")
    if set(df_match["source1_entity_id"]) != valid_s1: issues.append("Missing/Extra Source 1 IDs")
    cand_map = dict(zip(df_cand["source1_entity_id"], df_cand["candidate_entity_ids"]))
    for _, row in df_match.iterrows():
        s1 = row["source1_entity_id"]
        matches = [m.strip() for m in row["matched_entity_ids"].split(",")] if row["matched_entity_ids"] else []
        cands = [c.strip() for c in cand_map.get(s1, "").split(",")] if cand_map.get(s1, "") else []
        if len(matches) != len(set(matches)): issues.append(f"Dup matches in {s1}")
        for m in matches:
            if m not in valid_pool: issues.append(f"Invalid match ID {m} in {s1}")
            if m not in cands: issues.append(f"Match {m} not in candidates for {s1}")
    if issues:
        print("FAIL"); [print(f"- {i}") for i in issues[:10]]; sys.exit(1)
    print("PASS: Safe to submit!"); sys.exit(0)
if __name__ == "__main__":
    parser = argparse.ArgumentParser(); parser.add_argument("--matching", required=True); parser.add_argument("--candidate", required=True); parser.add_argument("--test-dir", required=True)
    args = parser.parse_args(); validate(args.matching, args.candidate, args.test_dir)
"""
with open("utils/validate_submission.py", "w") as f: f.write(validator_code)
!python3 utils/validate_submission.py --matching output/matching_results.tsv --candidate output/candidate_pairs.tsv --test-dir dataset/test

# 2. ZIP OUTPUTS
zip_name = "Final_Submission_Outputs.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zipf:
    zipf.write("output/matching_results.tsv", "output/matching_results.tsv")
    zipf.write("output/candidate_pairs.tsv", "output/candidate_pairs.tsv")
print(f"🎉 Success! {zip_name} created. You can now download and upload this to the platform.")

PASS: Safe to submit!
🎉 Success! Final_Submission_Outputs.zip created. You can now download and upload this to the platform.
